In [2]:
import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import Pipeline

NEGATION_WORDS = {"not", "no", "never", "n't", "cannot", "cant", "without"}
DEFAULT_STOP_WORDS = {
    "a", "an", "the", "is", "are", "was", "were", "be", "been", "being",
    "of", "in", "on", "at", "to", "for", "and", "or", "but", "with",
    "this", "that", "these", "those", "it", "its", "as", "so", "very",
    "i", "my", "me", "we", "our", "you", "your",
} - NEGATION_WORDS

def clean_text(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[^a-z0-9\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize(text: str) -> list:
    return text.split()

def remove_stop_words(tokens: list, stop_words: set = None) -> list:
    stop_words = stop_words if stop_words is not None else DEFAULT_STOP_WORDS
    return [t for t in tokens if t not in stop_words]

def preprocess(text: str, remove_stops: bool = True) -> str:
    cleaned = clean_text(text)
    tokens = tokenize(cleaned)
    if remove_stops:
        tokens = remove_stop_words(tokens)
    return " ".join(tokens)

In [3]:
df = pd.read_csv('feedback.csv')
df['sentiment'].value_counts()

,count
sentiment,
negative,22
positive,13
neutral,6


In [4]:
def build_pipeline():
    return Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
        ("classifier", LogisticRegression(max_iter=1000)),
    ])

def train_sentiment_model(texts, labels, test_size=0.2, random_state=42):
    cleaned_texts = [preprocess(t) for t in texts]

    x_train, x_test, y_train, y_test = train_test_split(
        cleaned_texts, labels, test_size=test_size, random_state=random_state
    )

    model = build_pipeline()
    model.fit(x_train, y_train)

    predictions = model.predict(x_test)
    print("Sentiment model evaluation")
    print("-" * 40)
    print(classification_report(y_test, predictions, zero_division=0))
    print("Confusion matrix:")
    print(confusion_matrix(y_test, predictions))

    return model

def predict_sentiment(model, text):
    cleaned = preprocess(text)
    return model.predict([cleaned])[0]

In [5]:
model = train_sentiment_model(df['feedback'], df['sentiment'])

Sentiment model evaluation
----------------------------------------
              precision    recall  f1-score   support

    negative       0.43      0.60      0.50         5
     neutral       0.00      0.00      0.00         1
    positive       0.00      0.00      0.00         3

    accuracy                           0.33         9
   macro avg       0.14      0.20      0.17         9
weighted avg       0.24      0.33      0.28         9

Confusion matrix:
[[3 0 2]
 [1 0 0]
 [3 0 0]]


In [6]:
examples = [
    'The support team was excellent',
    'The application is not good',
    'The application was updated yesterday',
]

for text in examples:
    print(f'{text!r:45} -> {predict_sentiment(model, text)}')

'The support team was excellent'              -> positive
'The application is not good'                 -> negative
'The application was updated yesterday'       -> negative


In [7]:
import joblib
joblib.dump(model, 'sentiment_model.joblib')
print('Model saved.')

Model saved.
